# Data Catalog

This notebook shows what tracked datasets already exist in the repo, how fresh they are, and what the three research marts look like.


In [11]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

BASE_DIR = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
SRC_DIR = BASE_DIR / "src"
for path in (BASE_DIR, SRC_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print(f"Base dir: {BASE_DIR}")


Base dir: /Users/henrywzh/Desktop/Quant/alternative-data


In [12]:
from research_data.api import (
    build_daily_provider_economics,
    build_frontier_model_registry,
    build_weekly_openrouter_usage,
    catalog,
)

catalog_df = catalog(base_dir=BASE_DIR)
print(catalog_df[["dataset_kind", "dataset_id", "domain", "row_count", "first_date", "latest_date"]].to_string(index=False))
catalog_df.head(12)


dataset_kind                          dataset_id               domain  row_count                  first_date                 latest_date
        mart            daily_provider_economics             research       8659                  2026-01-16                  2026-04-17
        mart             frontier_model_registry             research        289                  2023-03-21                  2026-04-08
        mart             weekly_openrouter_usage             research       1550                  2025-03-30                  2026-04-12
      source                      llm_benchmarks          ai_frontier        289                  2023-03-21                  2026-04-08
      source              app_metadata_snapshots                 apps         26                  2026-04-05                  2026-04-19
      source       app_top_models_daily_snapshot                 apps        520                  2026-04-05                  2026-04-19
      source                     app_usag

,dataset_kind,dataset_id,label,domain,source_path,source_format,row_count,primary_date_column,metric_column,first_date,latest_date,latest_scraped_at,duplicate_rows,missing_columns,latest_manifest_run_id,latest_manifest_scraped_at
0,mart,daily_provider_economics,Daily Provider Economics,research,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,8659,usage_date,estimated_revenue,2026-01-16,2026-04-17,None,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
1,mart,frontier_model_registry,Frontier Model Registry,research,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,289,release_date,frontier_score,2023-03-21,2026-04-08,None,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
2,mart,weekly_openrouter_usage,Weekly OpenRouter Usage,research,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,1550,week_start_date,metric_value,2025-03-30,2026-04-12,None,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
3,source,llm_benchmarks,LLM Benchmarks,ai_frontier,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,289,release_date,gpqa,2023-03-21,2026-04-08,2026-04-15T23:07:13.818716Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
4,source,app_metadata_snapshots,App Metadata,apps,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,26,scrape_date,None,2026-04-05,2026-04-19,2026-04-19T03:52:25.404691Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
5,source,app_top_models_daily_snapshot,App Top Models,apps,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,520,snapshot_date,total_tokens,2026-04-05,2026-04-19,2026-04-19T03:52:25.404691Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
6,source,app_usage_daily,App Usage Daily,apps,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,959,usage_date,total_tokens,2026-03-06,2026-04-19,2026-04-19T03:52:25.404691Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
7,source,apps_global_ranking_snapshots,Global App Rankings,apps,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,9000,snapshot_date,tokens,2026-04-05,2026-04-19,2026-04-19T03:52:25.404691Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
8,source,apps_trending_snapshots,Trending Apps,apps,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,120,snapshot_date,tokens,2026-04-05,2026-04-19,2026-04-19T03:52:25.404691Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z
9,source,raw_aws_spot_price_history,AWS Spot Pricing,compute_availability,/Users/henrywzh/Desktop/Quant/alternative-data...,parquet,140,price_timestamp,spot_price,2026-04-14T08:59:47+00:00,2026-04-15T22:46:30+00:00,2026-04-15T22:56:37.109991Z,0,,20260417T205441Z-7ef6cf1b,2026-04-17T20:54:41.324384Z


## Build Or Load Research Marts

The calls below use the persisted mart files when they already exist, and build them if they are missing.


In [13]:
weekly_usage = build_weekly_openrouter_usage(base_dir=BASE_DIR, refresh=False)
daily_provider_economics = build_daily_provider_economics(base_dir=BASE_DIR, refresh=False)
frontier_registry = build_frontier_model_registry(base_dir=BASE_DIR, refresh=False)

print("weekly_openrouter_usage rows:", len(weekly_usage))
print("daily_provider_economics rows:", len(daily_provider_economics))
print("frontier_model_registry rows:", len(frontier_registry))


weekly_openrouter_usage rows: 1550
daily_provider_economics rows: 8659
frontier_model_registry rows: 289


In [14]:
mart_preview = {
    "weekly_openrouter_usage": weekly_usage.head(5),
    "daily_provider_economics": daily_provider_economics.head(5),
    "frontier_model_registry": frontier_registry.head(5),
}
mart_preview


{'weekly_openrouter_usage':   week_start_date time_grain dataset_source entity_type   entity_id entity_name parent_entity_id parent_entity_name      metric_name metric_unit  \
 0      2025-03-30       week   market_share      author      google      google             <NA>               <NA>  token_share_pct       share   
 1      2025-03-30       week   market_share      author   anthropic   anthropic             <NA>               <NA>  token_share_pct       share   
 2      2025-03-30       week   market_share      author      openai      openai             <NA>               <NA>  token_share_pct       share   
 3      2025-03-30       week   market_share      author    deepseek    deepseek             <NA>               <NA>  token_share_pct       share   
 4      2025-03-30       week   market_share      author  openrouter  openrouter             <NA>               <NA>  token_share_pct       share   
 
    metric_value  rank category_slug                      source_url         